In [40]:
# Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate, cross_val_predict
from sklearn.preprocessing import StandardScaler

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    GradientBoostingClassifier
)
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    ConfusionMatrixDisplay
)

import matplotlib.pyplot as plt

In [41]:
games = pd.read_csv("../data/processed/game_lifecycle_analyzed.csv")

games

,app_id,title,launch_price,reviews_total,review_score,tags,baseline_players,relative_6m,relative_12m,relative_24m,early_trend_slope,month_6_vs_month_1,early_volatility,early_peak_to_average,lifecycle_category,is_multiplayer
0,730,Counter-Strike: Global Offensive,14.99,7382695,0.88,"FPS, Shooter, Multiplayer, Competitive, Action...",13624.776667,1.210952,1.813688,9.060906,0.046684,1.085745,0.158598,1.177936,Growing,True
1,2870,X Rebirth,49.99,5102,0.41,"Simulation, Space, Sci fi, Action, Space Sim, ...",617.486667,0.594242,0.605125,0.315354,-0.176420,0.467198,0.602581,2.149940,Declining,False
2,4920,Natural Selection 2,9.99,8198,0.85,"Multiplayer, Strategy, FPS, Team Based, Action...",2238.970000,0.493038,0.330320,0.165360,-0.179528,0.324727,0.387130,1.605843,Declining,True
3,8870,BioShock Infinite,59.99,98966,0.93,"FPS, Story Rich, Action, Singleplayer, Steampu...",3600.803333,0.422417,0.548959,0.295640,-0.264835,0.173542,0.829654,2.649201,Declining,False
4,10270,Disciples III: Reincarnation,29.99,1066,0.69,"RPG, Strategy, Turn Based Strategy, Turn Based...",80.743333,0.653057,0.668868,0.615572,-0.146955,0.398816,0.375316,1.736611,Sustained,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2537,799640,Dungeon Munchies,19.99,5597,0.97,"Indie, Action, Pixel Graphics, Adventure, RPG,...",54.923333,0.502033,0.251684,0.152212,-0.197041,0.312445,0.771824,2.533049,Declining,False
2538,800270,Terraforming Mars,24.99,4029,0.70,"Strategy, Board Game, Space, Tabletop, Turn Ba...",165.506667,1.013252,1.297269,2.531741,-0.062728,0.719030,0.180023,1.346088,Growing,True
2539,801480,Agent A: A puzzle in disguise,19.99,3864,0.95,"Logic, Point & Click, Hidden Object, Explorati...",11.000000,0.916970,0.942727,1.668182,-0.022287,0.465969,0.427646,1.639720,Growing,False
2540,801550,VAIL VR,19.99,694,0.76,"Early Access, VR, Shooter, Action, FPS, Multip...",44.466667,0.627511,0.501649,0.468966,-0.103055,0.531856,0.197713,1.197652,Declining,True


In [42]:
# Create multiplayer column
games["is_multiplayer"] = (games["tags"].str.contains("Multiplayer", case=False, na=False).astype(int))

games

,app_id,title,launch_price,reviews_total,review_score,tags,baseline_players,relative_6m,relative_12m,relative_24m,early_trend_slope,month_6_vs_month_1,early_volatility,early_peak_to_average,lifecycle_category,is_multiplayer
0,730,Counter-Strike: Global Offensive,14.99,7382695,0.88,"FPS, Shooter, Multiplayer, Competitive, Action...",13624.776667,1.210952,1.813688,9.060906,0.046684,1.085745,0.158598,1.177936,Growing,1
1,2870,X Rebirth,49.99,5102,0.41,"Simulation, Space, Sci fi, Action, Space Sim, ...",617.486667,0.594242,0.605125,0.315354,-0.176420,0.467198,0.602581,2.149940,Declining,0
2,4920,Natural Selection 2,9.99,8198,0.85,"Multiplayer, Strategy, FPS, Team Based, Action...",2238.970000,0.493038,0.330320,0.165360,-0.179528,0.324727,0.387130,1.605843,Declining,1
3,8870,BioShock Infinite,59.99,98966,0.93,"FPS, Story Rich, Action, Singleplayer, Steampu...",3600.803333,0.422417,0.548959,0.295640,-0.264835,0.173542,0.829654,2.649201,Declining,0
4,10270,Disciples III: Reincarnation,29.99,1066,0.69,"RPG, Strategy, Turn Based Strategy, Turn Based...",80.743333,0.653057,0.668868,0.615572,-0.146955,0.398816,0.375316,1.736611,Sustained,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2537,799640,Dungeon Munchies,19.99,5597,0.97,"Indie, Action, Pixel Graphics, Adventure, RPG,...",54.923333,0.502033,0.251684,0.152212,-0.197041,0.312445,0.771824,2.533049,Declining,0
2538,800270,Terraforming Mars,24.99,4029,0.70,"Strategy, Board Game, Space, Tabletop, Turn Ba...",165.506667,1.013252,1.297269,2.531741,-0.062728,0.719030,0.180023,1.346088,Growing,1
2539,801480,Agent A: A puzzle in disguise,19.99,3864,0.95,"Logic, Point & Click, Hidden Object, Explorati...",11.000000,0.916970,0.942727,1.668182,-0.022287,0.465969,0.427646,1.639720,Growing,0
2540,801550,VAIL VR,19.99,694,0.76,"Early Access, VR, Shooter, Action, FPS, Multip...",44.466667,0.627511,0.501649,0.468966,-0.103055,0.531856,0.197713,1.197652,Declining,1


In [43]:
# Compress baseline players as extremely skewed
games["log_baseline_players"] = np.log1p(games["baseline_players"])

In [57]:
# Shared evaluation helpers
outer_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
inner_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
scoring = {"accuracy": "accuracy", "macro_f1": "f1_macro"}

def candidate_searches():
    return {
        "Logistic Regression": GridSearchCV(
            Pipeline([("scaler", StandardScaler()),
                      ("model", LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42))]),
            {"model__C": [0.1, 1, 10]}, cv=inner_cv, scoring="f1_macro", n_jobs=-1
        ),
        "KNN": GridSearchCV(
            Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier())]),
            {"model__n_neighbors": [3, 5, 9]}, cv=inner_cv, scoring="f1_macro", n_jobs=-1
        ),
        "SVM": GridSearchCV(
            Pipeline([("scaler", StandardScaler()),
                      ("model", SVC(kernel="rbf", class_weight="balanced", probability=True, random_state=42))]),
            {"model__C": [0.5, 1, 2]}, cv=inner_cv, scoring="f1_macro", n_jobs=-1
        ),
        "Random Forest": GridSearchCV(
            RandomForestClassifier(class_weight="balanced", random_state=42),
            {"n_estimators": [200], "max_depth": [6, 8, None], "min_samples_leaf": [2, 5]},
            cv=inner_cv, scoring="f1_macro", n_jobs=-1
        ),
    }

def evaluate_feature_set(X, label):
    results = []
    for name, search in candidate_searches().items():
        scores = cross_validate(search, X, y, cv=outer_cv, scoring=scoring, n_jobs=-1)
        results.append({
            "Model": name,
            "Accuracy (mean ± std)": f"{scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}",
            "Macro F1 (mean ± std)": f"{scores['test_macro_f1'].mean():.3f} ± {scores['test_macro_f1'].std():.3f}",
            "Macro F1 mean": scores['test_macro_f1'].mean(),
        })
    results = pd.DataFrame(results).sort_values("Macro F1 mean", ascending=False).reset_index(drop=True)
    print(f"{label} prediction — nested stratified CV")
    return results

In [58]:
# Features available at each prediction
features_6m = ["launch_price", "log_baseline_players", "is_multiplayer", "relative_6m", "early_trend_slope", "month_6_vs_month_1", "early_volatility", "early_peak_to_average",]
features_12m = features_6m + ["relative_12m"]
X_6m = games[features_6m]
X_12m = games[features_12m]
y = games["lifecycle_category"]

results_6m = evaluate_feature_set(X_6m, "6-month")
results_6m

6-month prediction — nested stratified CV


,Model,Accuracy (mean ± std),Macro F1 (mean ± std),Macro F1 mean
0,Random Forest,0.563 ± 0.023,0.530 ± 0.022,0.530343
1,SVM,0.542 ± 0.014,0.520 ± 0.015,0.520337
2,Logistic Regression,0.534 ± 0.018,0.508 ± 0.021,0.507646
3,KNN,0.557 ± 0.015,0.480 ± 0.027,0.479655


In [59]:
results_12m = evaluate_feature_set(X_12m, "12-month")
results_12m

12-month prediction — nested stratified CV


,Model,Accuracy (mean ± std),Macro F1 (mean ± std),Macro F1 mean
0,Random Forest,0.675 ± 0.017,0.646 ± 0.021,0.645984
1,SVM,0.655 ± 0.022,0.623 ± 0.026,0.622721
2,Logistic Regression,0.653 ± 0.019,0.619 ± 0.021,0.618859
3,KNN,0.614 ± 0.010,0.552 ± 0.012,0.551784


In [47]:
# Final 6-month model
model_6m = GridSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),
    {
        "n_estimators": [200],
        "max_depth": [6, 8, None],
        "min_samples_leaf": [2, 5],
    },
    cv=inner_cv,
    scoring="f1_macro",
    n_jobs=-1
).fit(X_6m, y).best_estimator_

# Final 12-month model
model_12m = GridSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),
    {
        "n_estimators": [200],
        "max_depth": [6, 8, None],
        "min_samples_leaf": [2, 5],
    },
    cv=inner_cv,
    scoring="f1_macro",
    n_jobs=-1
).fit(X_12m, y).best_estimator_

In [48]:
# Every OOF prediction is made by a model that did not train on that game's row.
oof_6m_search = candidate_searches()["Random Forest"]
oof_12m_search = candidate_searches()["Random Forest"]
oof_6m_pred = cross_val_predict(oof_6m_search, X_6m, y, cv=outer_cv, method="predict", n_jobs=-1)
oof_6m_proba = cross_val_predict(oof_6m_search, X_6m, y, cv=outer_cv, method="predict_proba", n_jobs=-1)
oof_12m_pred = cross_val_predict(oof_12m_search, X_12m, y, cv=outer_cv, method="predict", n_jobs=-1)
oof_12m_proba = cross_val_predict(oof_12m_search, X_12m, y, cv=outer_cv, method="predict_proba", n_jobs=-1)

oof_predictions = games[["app_id", "title", "lifecycle_category"]].copy()
oof_predictions = oof_predictions.rename(columns={"lifecycle_category": "actual_lifecycle"})
oof_predictions["predicted_6m"] = oof_6m_pred
oof_predictions["confidence_6m"] = oof_6m_proba.max(axis=1)
oof_predictions["predicted_12m"] = oof_12m_pred
oof_predictions["confidence_12m"] = oof_12m_proba.max(axis=1)

oof_predictions.to_csv("../data/processed/lifecycle_oof_predictions.csv", index=False)
oof_predictions.head()

,app_id,title,actual_lifecycle,predicted_6m,confidence_6m,predicted_12m,confidence_12m
0,730,Counter-Strike: Global Offensive,Growing,Growing,0.775260,Growing,0.847173
1,2870,X Rebirth,Declining,Declining,0.483809,Sustained,0.438346
2,4920,Natural Selection 2,Declining,Declining,0.510851,Declining,0.808117
3,8870,BioShock Infinite,Declining,Declining,0.590436,Sustained,0.542913
4,10270,Disciples III: Reincarnation,Sustained,Sustained,0.485396,Sustained,0.645439


In [49]:
# Save models
import joblib
import os

joblib.dump(
    model_6m,
    "../models/lifecycle_6m.pkl"
)

joblib.dump(
    model_12m,
    "../models/lifecycle_12m.pkl"
)

['../models/lifecycle_12m.pkl']